# Evidently lesson: worked solutions to the exercises

Run this notebook top to bottom (about 4 minutes). Everything is written to **temporary folders**
(simulated data, Evidently reports), so it changes nothing of yours. Each exercise shows the question, the code
that answers it, and an **Answer** based on the numbers printed above it.
Participants: if you could not solve an exercise, or want to check your own answer, just run the cells.

In [ ]:
import sys, tempfile, warnings, json
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, "../03-mlflow")        # reuse the MLflow lesson's helpers (read only)
import emr_model as em

import numpy as np
import pandas as pd
from evidently import DataDefinition, Dataset, Report
from evidently.metrics import MissingValueCount, ValueDrift
from evidently.tests import lte
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 3)
DATA = Path(tempfile.mkdtemp(prefix="evidently_solutions_"))     # all data goes here, not into the repo
print("temporary folder:", DATA)

# Same setup as monitoring.ipynb: the 2024 model, and a helper that scores a month of a scenario
em.simulate("2024-01", 12, out=DATA / "raw")
history = em.load_months("2024-01", "2024-12", raw=DATA / "raw")
model = em.build_model("logistic").fit(history[em.FEATURES], history[em.LABEL])
history["risk"] = model.predict_proba(history[em.FEATURES])[:, 1]

def scored(month, scenario="baseline"):
    folder = DATA / scenario
    em.simulate(month, 1, scenario=scenario, out=folder)
    df = em.load_months(month, month, raw=folder)
    df["risk"] = model.predict_proba(df[em.FEATURES])[:, 1]
    return df

LEAN = {6, 7, 8, 9}
num = [c for c in em.NUMERIC if c != "lean_season"] + ["risk"]
definition = DataDefinition(numerical_columns=num, categorical_columns=em.CATEGORICAL)
cols = num + em.CATEGORICAL

def reference_for(month):
    """Same-season reference: 2024's lean months for a lean month, the other months otherwise."""
    return history[history["lean_season"] == int(int(month[5:7]) in LEAN)]

print(f"model trained on {len(history)} visits from 2024")

### The gate from monitoring.ipynb, made configurable

Same gate as section 3 of the lesson. Three knobs for the exercises: `missing_checks` (exercise 1),
`risk_limit` (exercise 3) and `reference` (exercise 4).

In [ ]:
def monitoring_gate(current, month, reference=None, risk_limit=0.1, missing_checks=True):
    reference = reference_for(month) if reference is None else reference
    metrics = [
        ValueDrift(column="risk", tests=[lte(risk_limit)]),
        ValueDrift(column="region", tests=[lte(0.1)]),
        ValueDrift(column="muac_cm", tests=[lte(0.1)]),
        ValueDrift(column="whz", tests=[lte(0.1)]),
    ]
    if missing_checks:
        metrics += [MissingValueCount(column="weight_kg", share_tests=[lte(0.05)]),
                    MissingValueCount(column="height_cm", share_tests=[lte(0.05)])]
    snap = Report(metrics).run(Dataset.from_pandas(current[cols], data_definition=definition),
                               Dataset.from_pandas(reference[cols], data_definition=definition))
    failed = [t["name"].split(":")[0] for t in snap.dict()["tests"] if str(t["status"]).endswith("FAIL")]
    return snap, failed

def drift_scores(snap):
    """Drift score per column from a gate report."""
    return {m["config"]["column"]: m["value"] for m in snap.dict()["metrics"]
            if m["config"]["type"].endswith("ValueDrift")}

scenarios = ["baseline", "new_regions", "muac_mm", "missing_anthro", "concept_shift", "messy_export"]

## Exercise 1

**Missing values the naive way.** Remove the two `MissingValueCount` checks from the gate and rerun the
scenarios. Which scenario now passes, and why is that dangerous?

In [ ]:
# Run every scenario (January 2025) through the gate with and without the two missing-value checks
rows = {}
for sc in scenarios:
    cur = scored("2025-01", sc)
    _, with_checks = monitoring_gate(cur, "2025-01")
    snap, without = monitoring_gate(cur, "2025-01", missing_checks=False)
    rows[sc] = {"with missing checks": ", ".join(with_checks) or "ok",
                "without missing checks": ", ".join(without) or "ok"}
pd.DataFrame(rows).T

No scenario flips to "ok" in this table: `missing_anthro` still alerts, but only through `risk`. That is luck, not design:
the imputer fills the missing weights and the risk score moves. The next cell shows how much missingness it takes before
that luck runs out.

In [ ]:
# Blank out weight, height and WHZ in a random share of a NORMAL month, then run both gates.
rng = np.random.default_rng(1)
base = scored("2025-01")
rows = []
for share in [0, 0.1, 0.2, 0.3, 0.5, 0.7]:
    cur = base.copy()
    gone = rng.random(len(cur)) < share
    cur.loc[gone, ["weight_kg", "height_cm", "whz"]] = np.nan
    cur["risk"] = model.predict_proba(cur[em.FEATURES])[:, 1]          # the model imputes silently
    snap, naive = monitoring_gate(cur, "2025-01", missing_checks=False)
    _, full = monitoring_gate(cur, "2025-01")
    rows.append({"missing weight/height/WHZ": f"{share:.0%}",
                 "risk drift score": drift_scores(snap)["risk"],
                 "naive gate (no missing checks)": "ALERT" if naive else "ok",
                 "full gate": "ALERT" if full else "ok",
                 "ROC-AUC (outcomes known later)": roc_auc_score(cur[em.LABEL], cur["risk"])})
pd.DataFrame(rows).set_index("missing weight/height/WHZ")

**Answer.** In this data **no scenario flips to "ok"**: with the checks removed, `missing_anthro` still alerts, but only with
`Value drift for risk`, because the model quietly fills in the missing weights and the risk scores move.
That is luck. The sweep shows how thin the luck is: with **10%, 20% and 30%** of weight, height and WHZ missing, the
naive gate says **ok** (risk drift 0.020, 0.040, 0.054 is under the 0.1 limit) while the full gate alerts, and ROC-AUC has already
slipped from 0.805 to 0.793 at 20% missing. Only from about 50% missing does the risk score move enough to trip the naive gate.

Why it is dangerous: Evidently's drift tests compare the values that are present, so missing data is invisible to them. A gate
without the missing-value checks passes batches where a third of the measurements are gone and the model is running on
imputed medians, and when it does fire, it names the wrong cause (a risk shift, not a broken scale or an export change).
Keep an explicit missingness check for every input the model depends on.

## Exercise 2

**Earlier detection of concept shift?** Nothing in the inputs changed. Is there any label-free signal you could
add (e.g. referral counts, clinic-reported admissions)? What would the EMR need to record?

The simulator has **no** referral or admission column, so this cell **simulates one** (marked SIMULATED below).
The assumptions are ours, not data from the EMR:

- a child who goes on to develop GAM is admitted to treatment at the clinic with probability 0.6 (care-seeking),
- on a random day within the 90 days after the visit,
- the clinic's admission register records it (facility, date, child id).

The signal: admissions seen in the **first 30 days** after the month's visits, divided by the admissions the model
expects (the sum of the risk scores). If the ratio jumps compared with normal months, the world changed, and we
know after about a month rather than three. Strictly this is an early *outcome* proxy, not label-free in the
purest sense: the point is that it needs only counts from the admission register, not the 90-day follow-up of each child.

In [ ]:
# SIMULATED: clinic admissions (not a column in the EMR extract).
CARE_SEEKING, EARLY_DAYS = 0.6, 30
adm_rng = np.random.default_rng(7)

def early_admissions(df):
    """SIMULATED admission register: how many of this month's visits were admitted within EARLY_DAYS."""
    developed = df[em.LABEL] == 1                                     # the hidden truth in the simulator
    admitted = developed & (adm_rng.random(len(df)) < CARE_SEEKING)
    day = adm_rng.integers(1, 91, size=len(df))                       # days after the visit
    return int((admitted & (day <= EARLY_DAYS)).sum())

months = {"2025-01": "baseline", "2025-02": "baseline", "2025-03": "baseline",
          "2025-04": "concept_shift", "2025-05": "concept_shift", "2025-06": "concept_shift"}
rows = []
for month, sc in months.items():
    cur = scored(month, sc)
    exp = cur["risk"].sum() * CARE_SEEKING * EARLY_DAYS / 90          # admissions the model expects by day 30
    rows.append({"month": month, "(hidden truth)": sc, "expected by day 30": exp,
                 "admissions by day 30": early_admissions(cur)})
adm = pd.DataFrame(rows).set_index("month")
adm["ratio"] = adm["admissions by day 30"] / adm["expected by day 30"]
normal = adm.loc["2025-01":"2025-03", "ratio"]
adm["alert"] = adm["ratio"] > normal.mean() + 3 * normal.std()    # 3 standard deviations above the normal months
adm

In [ ]:
# What does the full 90-day follow-up say for the same months? (This is the late, label-based signal.)
late = {m: scored(m, sc)[em.LABEL].mean() / scored(m, sc)["risk"].mean() for m, sc in months.items()}
print("observed/predicted after 90 days:", {m: round(v, 2) for m, v in late.items()})
print(f"early-window ratio, normal months: {normal.mean():.2f} +/- {normal.std():.2f}")

**Answer.** Yes, if the clinics' own counts can be linked back to the visits. In the simulation (assumptions above, nothing here comes
from the EMR) the ratio of admissions seen by day 30 to admissions expected is **0.78 to 0.92 in the three normal months**
and **2.0 to 2.2 from April**, when the concept shift starts, so the alert fires on the first affected month, about **one month after
the month's visits instead of three** (the 90-day observed/predicted gives 2.13, 2.23, 1.91 for the same months).
It works because the shift changes how many children actually fall ill, which shows up in the admission register long before
we have each child's 90-day outcome. Limits: the ratio depends on the care-seeking rate, so compare with normal months rather than
with 1.0; 30-day counts are small (about 35 expected per month, so keep the threshold wide); and a change in
referral or admission policy would look like a shift.

What the EMR would need to record: (1) **admission or referral events** for the programme (date, facility, child id, type such as
SAM/MAM treatment, referral to stabilisation centre), linked to the child's earlier visits by `child_id`; (2) a **reliable date** for each
event, and a rule for when a facility's register counts as complete for the month (otherwise missing reports look like a drop);
(3) optionally the programme context that explains shifts, such as a food-distribution status flag per district. Other label-free
signals worth monitoring: referral counts from other services (outpatient treatment programme, community health workers), and
the prediction-drift score, which stayed quiet here.

## Exercise 3

**Thresholds.** Lower the risk-drift threshold to 0.05. Do normal months start to alert? What does a false alarm
cost the team each month?

In [ ]:
# Twelve NORMAL (baseline) months of 2025, each against its same-season reference
rows = []
for i in range(1, 13):
    month = f"2025-{i:02d}"
    cur = scored(month)
    snap, failed_05 = monitoring_gate(cur, month, risk_limit=0.05)
    _, failed_10 = monitoring_gate(cur, month, risk_limit=0.10)
    rows.append({"month": month, "risk drift score": drift_scores(snap)["risk"],
                 "gate at 0.10": "ALERT" if failed_10 else "ok", "gate at 0.05": "ALERT" if failed_05 else "ok"})
normal_months = pd.DataFrame(rows).set_index("month")
normal_months

In [ ]:
# Real problems for comparison: the risk drift score of each problem scenario
prob = {sc: drift_scores(monitoring_gate(scored("2025-03" if sc == "new_regions" else "2025-01", sc),
                                         "2025-03" if sc == "new_regions" else "2025-01")[0])["risk"]
        for sc in ["new_regions", "muac_mm", "missing_anthro", "messy_export"]}
print("normal months:   highest risk drift score = %.3f" % normal_months["risk drift score"].max())
print("problem months: ", {k: round(v, 3) for k, v in prob.items()})

In [ ]:
# The batches above have 2000 visits. A district with 500 visits a month is noisier: draw 500-visit
# batches from normal months and see how often a threshold of 0.05 or 0.10 would fire.
rng = np.random.default_rng(3)
scores = []
for month in [f"2025-{i:02d}" for i in range(1, 13)]:
    cur = scored(month)
    for _ in range(3):
        small = cur.sample(500, random_state=int(rng.integers(1_000_000)))
        snap = Report([ValueDrift(column="risk")]).run(
            Dataset.from_pandas(small[cols], data_definition=definition),
            Dataset.from_pandas(reference_for(month)[cols], data_definition=definition))
        scores.append(drift_scores(snap)["risk"])
scores = np.array(scores)
print(f"{len(scores)} normal batches of 500 visits: median score {np.median(scores):.3f}, max {scores.max():.3f}")
print(f"share that would alert at 0.05: {(scores > 0.05).mean():.0%}    at 0.10: {(scores > 0.10).mean():.0%}")

# Cost of a false alarm: an ASSUMPTION to replace with your own numbers
HOURS_PER_ALARM = 3            # open report, check the EMR export, ask the clinics, write it up
per_year = (scores > 0.05).mean() * 12
print(f"at 0.05 with 500-visit batches: ~{per_year:.1f} false alarms a year, ~{per_year * HOURS_PER_ALARM:.0f} analyst hours")

**Answer.** **No, not with 2000-visit months**: all twelve normal months stay under 0.05 (highest 0.044 in October), so the gate is quiet at both
limits. But the margin is thin, and it disappears with fewer visits: in 500-visit batches (a district rather than a country) the median score is already 0.050,
so **half of the normal batches would alert at 0.05**, against 3% at 0.10. Meanwhile real problems score far higher
(new_regions 0.32, muac_mm 0.73, missing_anthro 0.18, messy_export 0.14), so 0.10 loses nothing here and 0.05 only buys noise.

The cost of a false alarm: someone opens the report, checks the EMR export and calls the clinics, and finds nothing. With our assumption of
3 hours each, 6 false alarms a year is about 18 analyst hours, plus the bigger cost: people stop reading the alerts, and the one real alarm gets
ignored. Choose the threshold from how normal months behave at your batch size (run the check above on your own history), not by feel.

## Exercise 4

**Per-group drift.** Run the gate separately for Afar + Somali visits in March. Is the change concentrated there?
(Hint: filter both current and reference.)

In [ ]:
march = scored("2025-03", "new_regions")
ref = reference_for("2025-03")
groups = {"Afar + Somali": lambda d: d["region"].isin(["Afar", "Somali"]),
          "all other regions": lambda d: ~d["region"].isin(["Afar", "Somali"])}

rows = {}
snap_all, failed_all = monitoring_gate(march, "2025-03")
rows["everyone"] = {"visits (March)": len(march), "visits (reference)": len(ref), **drift_scores(snap_all),
                    "mean risk March": march["risk"].mean(), "mean risk reference": ref["risk"].mean(),
                    "failed checks": ", ".join(failed_all) or "-"}
for name, pick in groups.items():
    cur_g, ref_g = march[pick(march)], ref[pick(ref)]                   # filter BOTH current and reference
    snap, failed = monitoring_gate(cur_g, "2025-03", reference=ref_g)
    rows[name] = {"visits (March)": len(cur_g), "visits (reference)": len(ref_g), **drift_scores(snap),
                  "mean risk March": cur_g["risk"].mean(), "mean risk reference": ref_g["risk"].mean(),
                  "failed checks": ", ".join(failed) or "-"}
pd.DataFrame(rows).T

In [ ]:
# What changed is the MIX of visits. Share of visits from Afar + Somali:
share = lambda d: d["region"].isin(["Afar", "Somali"]).mean()
print(f"Afar + Somali are {share(ref):.0%} of the reference visits and {share(march):.0%} of March's.")

**Answer.** **No, the change is not concentrated in Afar + Somali visits.** Run on their own (current and reference both filtered),
the group shows no failed check: risk drift 0.061 (under 0.1), region 0.020, and the mean risk is the same as before (0.167 vs 0.177).
The other regions look unchanged too (risk 0.038, mean risk 0.080 vs 0.080).
What changed is the **mix**: Afar + Somali go from 10% of the reference visits to 48% of March's, and they are higher-risk children,
so the batch as a whole drifts (risk 0.32, mean risk 0.089 to 0.122) while the model behaves the same inside each group.
So the March alert is real for workload and planning (more high-risk children to follow up), but it does not say the model is wrong for
these regions. Next steps: check performance and fairness per region once outcomes arrive. The Afar + Somali reference has only
1521 visits, so group-level scores are noisier than the overall ones.

## Exercise 5

**Wire it up.** Write a function that takes a month's file and returns `{"month", "status", "failed_checks", "mean_risk"}`
as a dict, ready for the Prefect pipeline to log or alert on.

In [ ]:
def check_month(path, model=model, risk_limit=0.1):
    """One month's EMR file -> a plain dict the pipeline can log or alert on."""
    path = Path(path)
    month = path.stem.removeprefix("emr_")                               # emr_2025-03.csv -> "2025-03"
    df = em.load_months(month, month, raw=path.parent)
    df["risk"] = model.predict_proba(df[em.FEATURES])[:, 1]
    _, failed = monitoring_gate(df, month, risk_limit=risk_limit)
    return {"month": month,
            "status": "ALERT" if failed else "ok",
            "failed_checks": failed,                                     # list of strings, JSON friendly
            "mean_risk": round(float(df["risk"].mean()), 4)}             # plain float, not numpy

# Try it on files from earlier: a normal month and the new_regions month
for f in [DATA / "baseline" / "emr_2025-01.csv", DATA / "new_regions" / "emr_2025-03.csv",
          DATA / "muac_mm" / "emr_2025-01.csv"]:
    result = check_month(f)
    print(json.dumps(result))                                            # serialises: safe to log or send

**Answer.** `check_month(path)` reads the month's file, scores it, runs the gate and returns only plain Python types
(`str`, `list[str]`, `float`), so it serialises to JSON as printed above: the normal January is `ok`, March (new regions) and the
millimetre-MUAC batch are `ALERT` with the failed checks listed. In the Prefect pipeline this replaces `drift_gate`: decorate it with
`@task`, log the dict (`get_run_logger().info(result)`), and branch on `result["status"]`, for example to send a notification
or put the dict straight into the monitoring-log row. Note it expects the file to have passed the validation gate (Great Expectations) first.

---
Cleaning up: the temporary folder is deleted in the next cell.

In [ ]:
import shutil
shutil.rmtree(DATA, ignore_errors=True)
print("temporary folder removed")